# Lab 03 — Bradley-Terry Reward Model (from scratch)

**Difficulty:** ★★☆ · **CORE** · Est. build 1–2 days

**Watch first**
- Lecture 2, part 2 (Reward models) — https://rlhfbook.com/course/
- Chapter 5, Bradley-Terry section — https://rlhfbook.com/c/05-reward-models.html
- Prerequisite lab: Lab 02 (your cleaned preference subset + near-tie export)

**Why this lab exists:** own the derivation of r(chosen) > r(rejected). You will hand-write the
BT loss, pool at the last non-pad token, and learn why reward *scale* is meaningless while
*margins* decide downstream best-of-N quality.

**Rules:** implement everything yourself before opening the answer key
(`rlhf-book/code/reward_models/base.py`, `train_preference_rm.py`).
Every code cell below is a TODO placeholder — no outputs are pre-computed.

## 0 · Setup

In [ ]:
# Colab / cluster installs (uncomment on a fresh runtime):
# %pip install -q torch transformers datasets accelerate wandb pyyaml

# TODO(Lab03): imports, seeds, device pick, point at configs/03_bradley_terry_rm.yaml
# HINT: RewardModelConfig in starter.py mirrors the YAML.

## 1 · Data: your cleaned Lab 02 subset

In [ ]:
# TODO(Lab03): load your cleaned UltraFeedback subset (2–5k pairs) from Lab 02,
# map its schema onto tokenize_pair(), build train/val loaders via build_dataloader().
# Sanity check: print one batch's key->shape table (right padding!).

## 2 · Model: backbone + Linear(hidden, 1) head

In [ ]:
# TODO(Lab03): instantiate BradleyTerryRewardModel (freeze_backbone=False first).
# Report trainable vs frozen parameter counts.
# Freezing is experiment arm A of §Experiments (freeze-backbone vs full-FT).

## 3 · Pooling unit-check — trace ONE pair end-to-end naming every tensor shape

Acceptance criterion for the whole lab: a narrated trace
`ids [B,T] -> hidden states [B,T,H] -> pooled [B,H] -> head [B,1] -> rewards [B]`
with concrete numbers pasted below.

In [ ]:
# TODO(Lab03): run one tiny batch through model.backbone with output_hidden_states=True,
# call pool_last_non_pad_token, verify shapes, then forward() end-to-end.
# Cross-check: decode only the last non-pad tokens to prove pooling picked real tokens,
# not pads (Lab 00's lesson).

## 4 · Hand-written BT loss + gradient sanity

Derivation required here (notebook markdown): P(chosen preferred) under BT, the NLL form,
and why your implementation must be numerically stable for large |scores|.

In [ ]:
# TODO(Lab03): implement bradley_terry_loss in starter.py, then:
#   - finite-difference or autograd-check the gradient wrt both scores;
#   - show loss unchanged under a constant shift applied to ALL scores (explain why).

## 5 · Metrics: accuracy · margins · calibration buckets · histograms

In [ ]:
# TODO(Lab03): implement pairwise_accuracy, mean_margin, accuracy_by_margin_bucket,
# compute_eval_metrics; run them at step 0 (untrained) as your baseline panel.
# Decide + document the tie convention for accuracy ONCE.

## 6 · Training loop (exportable script shares this config)

In [ ]:
# TODO(Lab03): wire run_training() — AdamW (trainable params only), grad accumulation,
# clip-grad-norm, linear warmup, AMP flag, JSONL metrics every eval_every_steps.
# Start with a ~100-step smoke run at max_length 512 before any long run.

## 7 · Evaluation figures — calibration & histograms

In [ ]:
# TODO(Lab03): plot (a) chosen/rejected reward histograms overlaid,
# (b) accuracy per margin bucket. Interpret in markdown below the plots:
# accuracy↑ while margins collapse ⇒ what does that mean for Lab 09?

## 8 · Experiments (README checklist drives these)

In [ ]:
# TODO(Lab03):
#   E1 freeze_backbone True vs False
#   E2 lr sweep {1e-6, 5e-6, 2e-5}
#   E3 data size {1k, 2k, 5k} vs pairwise accuracy
# Log each into runs/lab03_bt_rm/metrics.jsonl and plot together.

## 9 · Score your Lab 02 near-ties vs your human judgments

In [ ]:
# TODO(Lab03): score_lab02_near_ties(model, ...) → human-vs-RM verdict table.
# Flag every disagreement row; these feed later RTrust-calibration discussions.

## 10 · Wrap-up questions (answer inline in markdown)

1. Why sigmoid(diff)? Derive it from the BT likelihood once, in symbols.
2. Why is absolute reward scale meaningless? Show an invariance with a concrete tensor.
3. Accuracy↑ while margins collapse — what does that predict about best-of-N selection?
4. Debugging recap: accuracy stuck at 50% — which two places do you inspect first, and what exactly do you look for?
5. Margins exploding while accuracy stays flat — what is BT optimizing then?